# Gemma4EC chat: explain the predictions

For each reaction: the Gemma4EC prediction and its attention, the bonds that change, candidate enzymes with a curated
mechanism, and a TxGemma-9B-Chat explanation checked against those facts.

One-time setup (see `docs/chat.md`): `config.sh` with `BRENDA_FILE`, `METANETX_DIR`, `RXNMAPPER_PY` and
`GEMMA4EC_ADAPTER`; the RXNMapper environment; `bash chat/fetch_mcsa.sh`; access to
[txgemma-9b-chat](https://huggingface.co/google/txgemma-9b-chat). Needs a GPU (or set `CPU=1`, slow).

In [ ]:
import os, json, subprocess
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, Image, display

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
assert Path("config.sh").exists(), "copy config.sh.example to config.sh and fill in the paths"

## 1. Reactions

Preprocessing (`preprocess_smiles.py`, see the prediction notebook) is recommended here too.

In [ ]:
pd.DataFrame({"rxn_smiles": [
    "CCO.O=O>>CC=O.OO",
]}).to_csv("reactions.csv", index=False)
subprocess.run(["python", "preprocess_smiles.py", "reactions.csv", "reactions_prep.csv"], check=True)

## 2. Explain

Runs the four steps of `chat/explain.sh`: prediction + attention, reaction facts, candidates, chat.

In [ ]:
OUT = "chat_out"
subprocess.run(["bash", "chat/explain.sh", "reactions_prep.csv", OUT], check=True)

## 3. Read the explanations

In [ ]:
display(Markdown(Path(OUT, "explanations.md").read_text()))

## 4. Reaction centre and attention

Atoms whose bonds change, and where Gemma4EC attends, for reaction 1.

In [ ]:
RXN = 1
subprocess.run(["python", "figures/chat_reaction_panels.py", "--rxn", str(RXN), "--attn_dir", OUT,
                "--out", f"{OUT}/rxn{RXN:02d}"], check=True)
display(Image(f"{OUT}/rxn{RXN:02d}_reaction_centre.png"), Image(f"{OUT}/rxn{RXN:02d}_attention.png"))